# 10 — Wave Layer: PCA / K-means Market-Beta Research

**Strategy reference:** §8 (Wave), §18 (Permissions Matrix), §17 (Data).

**Research question:**  
Can a pruned universe of Binance + Oanda assets provide useful *Wave-layer
context* for BTC and ETH using correlation, lead/lag analysis, PCA, and
K-means clustering?

**Scope:**  
This is a **research notebook only**.  No production code is touched.  
Wave should not generate trades directly — it produces regime/context signals
that could later strengthen or weaken Ripple permissions.

---

| Section | Content |
|---|---|
| §0 | Setup & configuration |
| §1 | Data loading & diagnostics |
| §2 | Return construction |
| §3 | Universe pruning research |
| §4 | Universe selection |
| §5 | PCA research |
| §6 | K-means clustering |
| §7 | BTC/ETH residual model & forward tests |
| §8 | Wave feature proposal |
| §9 | Regime interpretation (research only) |

## §0 — Setup & configuration

In [ ]:
# ── Data-source configuration ──────────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment and set the two lines below
import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ──────────────────────────────────────────────────────────────────────────

import sys, importlib, warnings
from pathlib import Path

# Locate the project root (the directory containing schemas.py).
_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand
        break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# Reload notebooks package so on-disk changes are picked up without kernel restart.
import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)
from notebooks.utils import (
    load_ohlcv, list_ohlcv, configure_pandas, env_summary,
)

import notebooks.wave_factor_research as _wfr
importlib.reload(_wfr)
from notebooks.wave_factor_research import (
    score_universe, select_universe,
    rolling_pca, pca_loadings_table,
    kmeans_sweep, cluster_analytics,
    compute_residuals, forward_predictive_test, summarise_predictive_tests,
    wave_feature_table, regime_rules_table, WAVE_REGIME_RULES,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

configure_pandas()
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
# ── Research configuration ─────────────────────────────────────────────────
# All tunable parameters in one place.  Edit here before re-running.

CFG = {
    # Timeframes available in the repo.
    "timeframes":         ["1m", "5m", "15m"],
    # Active timeframe for this run.
    "active_timeframe":   "1m",
    # Date range (inclusive).  None = latest available.
    "from_date":          "2024-01-01",
    "to_date":            None,
    # Universe size target.
    "target_n_assets":    30,
    # Drop assets with coverage below this fraction.
    "min_coverage":       0.80,
    # Max gap (in bars) that is silently forward-filled.
    # Gaps larger than this are set to NaN and logged.
    "max_fill_bars":      3,
    # Return winsorisation threshold (±).
    "winsor_threshold":   0.005,
    # Rolling correlation window for scoring (bars).
    "rolling_corr_win":   60,
    # Lead/lag horizons in bars (1m timeframe → minutes).
    "lag_horizons":       [1, 5, 15, 30],
    # PCA settings.
    "pca_n_components":   5,
    "pca_windows":        [60, 240],   # rolling PCA look-back windows (bars)
    # K-means settings.
    "kmeans_k_range":     list(range(2, 9)),
    "kmeans_k_final":     4,
    "kmeans_n_init":      20,
    # Residual model rolling window (bars).
    "residual_window":    120,
    # Random seed — keeps notebook deterministic.
    "random_seed":        42,
    # Combined score weights (must sum to 1.0).
    "score_weights": {
        "max_abs_corr":    0.30,
        "stable_corr":     0.30,
        "predictive_corr": 0.30,
        "data_quality":    0.10,
    },
    # Core crypto bucket (always included if available).
    "core_crypto": [
        "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
        "DOGEUSDT", "ADAUSDT", "LINKUSDT", "AVAXUSDT",
    ],
    # Core macro / Oanda bucket (nearest available match used).
    "core_macro": [
        "NAS100_USD", "SPX500_USD", "XAU_USD",
        "EUR_USD", "USD_JPY", "GBP_USD", "AUD_USD", "BCO_USD",
    ],
    # Target columns for BTC and ETH.
    "btc_col": "BTCUSDT",
    "eth_col": "ETHUSDT",
}

np.random.seed(CFG["random_seed"])
print("Configuration loaded.")
print(f"  active_timeframe : {CFG['active_timeframe']}")
print(f"  from_date        : {CFG['from_date']}")
print(f"  target_n_assets  : {CFG['target_n_assets']}")
print(f"  random_seed      : {CFG['random_seed']}")

In [ ]:
# Snapshot of the active data-source configuration.
env_summary()

---
## §1 — Data loading & diagnostics

We inventory available assets for both Binance and Oanda, load each one,
align to a common timestamp index, and run explicit missingness diagnostics.
Assets below the coverage threshold are **dropped** and logged — no silent
forward-filling of large gaps.

In [ ]:
# ── Inventory: discover what is available ──────────────────────────────────
tf = CFG["active_timeframe"]

inv_binance = list_ohlcv("binance", tf)
inv_oanda   = list_ohlcv("oanda",   tf)

# Merge Parquet and HDF5 symbol lists (de-duplicate).
binance_syms = sorted(set(inv_binance.get("parquet", []) + inv_binance.get("hdf5", [])))
oanda_syms   = sorted(set(inv_oanda.get("parquet",   []) + inv_oanda.get("hdf5", [])))

print(f"Binance: {len(binance_syms)} symbols available at {tf}")
print(f"Oanda  : {len(oanda_syms)} symbols available at {tf}")

# TODO: if lists are empty, check that the collector has run and that
# DATA_STORE / S3_BUCKET are configured correctly (see env_summary above).

In [ ]:
# ── Load OHLCV and extract close prices ────────────────────────────────────

from_ts = int(pd.Timestamp(CFG["from_date"]).timestamp() * 1000)
to_ts   = (
    int(pd.Timestamp(CFG["to_date"]).timestamp() * 1000)
    if CFG["to_date"] else None
)

def _load_close(
    exchange: str,
    symbol: str,
    timeframe: str,
    from_ts: int,
    to_ts,
) -> pd.Series:
    """Load a single asset's OHLCV and return a close/mid price Series.

    Oanda data includes a 'spread' column; for Oanda we use
    (open + close) / 2 as the mid price to avoid bid/ask bias.
    """
    df = load_ohlcv(
        exchange, symbol, timeframe,
        from_ts=from_ts, to_ts=to_ts,
    )
    if df.empty:
        return pd.Series(dtype=float, name=symbol)
    if "spread" in df.columns:
        # Oanda mid-price
        price = (df["open"] + df["close"]) / 2.0
    else:
        price = df["close"]
    price.name = symbol
    return price


all_symbols  = [("binance", s) for s in binance_syms] + [("oanda", s) for s in oanda_syms]
price_series = []
load_errors  = []

for exchange, symbol in all_symbols:
    try:
        s = _load_close(exchange, symbol, tf, from_ts, to_ts)
        if len(s) > 0:
            price_series.append(s)
        else:
            load_errors.append((exchange, symbol, "empty series"))
    except Exception as exc:
        load_errors.append((exchange, symbol, str(exc)))

print(f"Loaded  : {len(price_series)} price series")
print(f"Errors  : {len(load_errors)}")
if load_errors:
    for exc_info in load_errors[:10]:
        print(f"  {exc_info[0]}/{exc_info[1]}: {exc_info[2][:80]}")
    if len(load_errors) > 10:
        print(f"  ... and {len(load_errors) - 10} more")

In [ ]:
# ── Align to common DatetimeIndex ──────────────────────────────────────────

# Build the raw price panel (outer join so we can see full coverage).
prices_raw = pd.concat(price_series, axis=1)
prices_raw.index = pd.to_datetime(prices_raw.index, utc=True)
prices_raw.sort_index(inplace=True)

print(f"Raw panel : {prices_raw.shape[0]:,} rows × {prices_raw.shape[1]} assets")
print(f"Date range: {prices_raw.index[0]} → {prices_raw.index[-1]}")

In [ ]:
# ── Missingness diagnostics ────────────────────────────────────────────────

tf_minutes = {
    "1m": 1, "5m": 5, "15m": 15, "30m": 30,
    "1h": 60, "4h": 240, "1d": 1440,
}.get(tf, 1)

total_rows = len(prices_raw)
coverage   = prices_raw.notna().sum() / total_rows
coverage_df = coverage.rename("coverage_frac").to_frame()
coverage_df["n_valid"]   = prices_raw.notna().sum()
coverage_df["n_missing"] = prices_raw.isna().sum()

below_threshold = coverage_df[coverage_df["coverage_frac"] < CFG["min_coverage"]]
above_threshold = coverage_df[coverage_df["coverage_frac"] >= CFG["min_coverage"]]

print(f"Coverage threshold : {CFG['min_coverage']:.0%}")
print(f"Assets above threshold : {len(above_threshold)}")
print(f"Assets below threshold (will be DROPPED) : {len(below_threshold)}")
if len(below_threshold) > 0:
    print("\nDropped assets:")
    print(below_threshold.sort_values("coverage_frac").to_string())

# Plot coverage distribution.
fig, ax = plt.subplots(figsize=(10, 3))
coverage_df["coverage_frac"].sort_values().plot.bar(ax=ax, width=1.0, color="steelblue")
ax.axhline(CFG["min_coverage"], color="red", ls="--", lw=1.2, label=f"min_coverage={CFG['min_coverage']:.0%}")
ax.set_xlabel("Asset")
ax.set_ylabel("Coverage fraction")
ax.set_title("Asset data coverage")
ax.set_xticks([])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Apply coverage filter & gap-filling policy ─────────────────────────────

good_assets = above_threshold.index.tolist()
prices      = prices_raw[good_assets].copy()

max_fill = CFG["max_fill_bars"]

# Forward-fill gaps of ≤ max_fill bars; leave larger gaps as NaN.
# We track how many cells were filled for the log.
nan_before = prices.isna().sum().sum()
for col in prices.columns:
    s = prices[col]
    # Mark consecutive NaN run lengths.
    nan_mask  = s.isna()
    run_id    = nan_mask.ne(nan_mask.shift()).cumsum()
    run_len   = nan_mask.groupby(run_id).transform("sum")
    # Only fill short gaps.
    fill_mask = nan_mask & (run_len <= max_fill)
    prices.loc[fill_mask, col] = s.ffill()[fill_mask]

nan_after = prices.isna().sum().sum()
print(f"Gap fill summary ({tf}, max_fill={max_fill} bars):")
print(f"  NaN before fill : {nan_before:,}")
print(f"  NaN after fill  : {nan_after:,}")
print(f"  Cells filled    : {nan_before - nan_after:,}")
print(f"  Residual NaN    : {nan_after:,} (large gaps — kept as NaN)")
print(f"\nFinal price panel : {prices.shape[0]:,} rows × {prices.shape[1]} assets")

---
## §2 — Return construction

We compute log returns for all assets, winsorise extreme values, and
standardise for PCA / K-means inputs.

In [ ]:
# ── Log returns ────────────────────────────────────────────────────────────

# log(P_t / P_{t-1}) — base transformation for all subsequent analysis.
log_returns = np.log(prices).diff()

# Drop the first row (NaN after diff) and any all-NaN columns.
log_returns = log_returns.iloc[1:]
log_returns.dropna(axis=1, how="all", inplace=True)

print(f"Log-return panel: {log_returns.shape[0]:,} rows × {log_returns.shape[1]} assets")

In [ ]:
# ── Winsorise extreme returns ──────────────────────────────────────────────

# Clip returns beyond ±threshold to remove outliers that distort correlations
# and PCA loadings without discarding the observation entirely.
threshold = CFG["winsor_threshold"]
returns_winsor = log_returns.clip(lower=-threshold, upper=threshold)

clipped_pct = (
    (log_returns.abs() > threshold).sum().sum()
    / log_returns.notna().sum().sum()
    * 100
)
print(f"Winsorisation threshold : ±{threshold:.3%}")
print(f"Fraction of values clipped: {clipped_pct:.3f}%")

In [ ]:
# ── Standardised returns (zero-mean, unit-variance per asset) ─────────────

# Used as input for PCA and K-means.  We standardise after winsorisation
# so that assets with different volatility levels contribute equally.
# NaN cells are left as NaN; sklearn handles them via row-drop in rolling_pca.

returns_std = (
    returns_winsor
    .sub(returns_winsor.mean())
    .div(returns_winsor.std().replace(0, np.nan))
)

# Separate BTC and ETH target series from the candidate pool.
btc_col = CFG["btc_col"]
eth_col = CFG["eth_col"]

if btc_col not in returns_winsor.columns:
    warnings.warn(
        f"BTC column '{btc_col}' not found in returns panel. "
        "Check that BTC data was collected and the column name matches CFG['btc_col']."
    )
if eth_col not in returns_winsor.columns:
    warnings.warn(
        f"ETH column '{eth_col}' not found in returns panel. "
        "Check that ETH data was collected and the column name matches CFG['eth_col']."
    )

btc_rets = returns_winsor.get(btc_col, pd.Series(dtype=float))
eth_rets = returns_winsor.get(eth_col, pd.Series(dtype=float))

candidate_cols = [
    c for c in returns_winsor.columns
    if c not in (btc_col, eth_col)
]
print(f"BTC column  : {btc_col}  ({btc_rets.notna().sum():,} valid rows)")
print(f"ETH column  : {eth_col}  ({eth_rets.notna().sum():,} valid rows)")
print(f"Candidates  : {len(candidate_cols)} assets")

In [ ]:
# ── Visualise BTC and ETH return distributions ─────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, [btc_col, eth_col], ["steelblue", "darkorange"]):
    s = returns_winsor[col].dropna()
    ax.hist(s, bins=100, color=color, alpha=0.7, edgecolor="none")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{col} log-return distribution ({tf})")
    ax.set_xlabel("Log return")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

---
## §3 — Universe pruning research

For every candidate asset we compute:
- Contemporaneous and absolute correlation to BTC/ETH.
- Rolling correlation mean and standard deviation.
- Correlation stability score: `stable_corr = mean(|rolling_corr|) / std(rolling_corr)`.
- Lead/lag correlations at horizons 1, 5, 15, 30 bars.
- A combined relevance score.

Scoring logic lives in `notebooks/wave_factor_research.py`.

In [ ]:
# ── Score all candidate assets ─────────────────────────────────────────────
# This may take a few minutes for large universes (~300 assets).

scores_df = score_universe(
    returns_winsor,
    btc_col=btc_col,
    eth_col=eth_col,
    cfg=CFG,
)

print(f"Scored {len(scores_df)} candidate assets.")
scores_df.head(10)

In [ ]:
# ── Ranked tables ─────────────────────────────────────────────────────────

N = 15

print(f"=== Top {N} by absolute BTC correlation ===")
display(scores_df.nlargest(N, "abs_corr_btc")[["corr_btc", "abs_corr_btc", "data_quality_score"]])

print(f"\n=== Top {N} by absolute ETH correlation ===")
display(scores_df.nlargest(N, "abs_corr_eth")[["corr_eth", "abs_corr_eth", "data_quality_score"]])

print(f"\n=== Top {N} by NEGATIVE correlation to BTC (potential hedges) ===")
display(scores_df.nsmallest(N, "corr_btc")[["corr_btc", "corr_eth", "data_quality_score"]])

print(f"\n=== Top {N} by correlation stability ===")
display(scores_df.nlargest(N, "stable_corr_score")[["stable_corr_score", "roll_mean_corr", "roll_std_corr"]])

print(f"\n=== Top {N} by predictive lead/lag correlation ===")
display(scores_df.nlargest(N, "predictive_corr_score")[["predictive_corr_score", "corr_btc", "corr_eth"]])

print(f"\n=== Top {N} by combined relevance score ===")
display(scores_df.head(N)[["combined_score", "abs_corr_btc", "abs_corr_eth", "stable_corr_score", "predictive_corr_score"]])

In [ ]:
# ── Lead/lag correlation heatmap (top 20 assets) ───────────────────────────

top20 = scores_df.head(20)
lag_cols_btc = [f"lag_corr_{h}m_btc" for h in CFG["lag_horizons"]]
lag_cols_eth = [f"lag_corr_{h}m_eth" for h in CFG["lag_horizons"]]
lag_all = lag_cols_btc + lag_cols_eth
available_lag_cols = [c for c in lag_all if c in top20.columns]

if available_lag_cols:
    fig, ax = plt.subplots(figsize=(14, 6))
    hm_data = top20[available_lag_cols].fillna(0)
    im = ax.imshow(hm_data.values, aspect="auto", cmap="RdBu", vmin=-0.5, vmax=0.5)
    ax.set_xticks(range(len(available_lag_cols)))
    ax.set_xticklabels(available_lag_cols, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(hm_data)))
    ax.set_yticklabels(hm_data.index, fontsize=8)
    ax.set_title("Lead/lag correlations — top 20 assets by combined score")
    plt.colorbar(im, ax=ax, fraction=0.02)
    plt.tight_layout()
    plt.show()

---
## §4 — Universe selection

We build the final ~30-asset research universe using the priority logic:
1. Anchor: BTC and ETH (always).
2. Core crypto bucket (if available).
3. Core macro / Oanda bucket (nearest match).
4. Fill remaining slots by `combined_score` descending.

In [ ]:
# ── Select final universe ──────────────────────────────────────────────────

all_available = list(returns_winsor.columns)

universe_df = select_universe(
    scores_df=scores_df,
    all_available=all_available,
    cfg=CFG,
    btc_col=btc_col,
    eth_col=eth_col,
)

print(f"Selected universe: {len(universe_df)} assets")
display(universe_df)

In [ ]:
# ── Build universe return and standardised-return panels ───────────────────

universe_syms = universe_df["symbol"].tolist()
# Filter to columns that exist in the return panel.
universe_syms = [s for s in universe_syms if s in returns_winsor.columns]

uni_returns   = returns_winsor[universe_syms].copy()
uni_returns_std = returns_std[universe_syms].copy()

# Drop rows where BTC or ETH is NaN (required for factor model).
anchor_mask = uni_returns[[btc_col, eth_col]].notna().all(axis=1)
uni_returns     = uni_returns[anchor_mask]
uni_returns_std = uni_returns_std[anchor_mask]

print(f"Universe return panel: {uni_returns.shape[0]:,} rows × {uni_returns.shape[1]} assets")

In [ ]:
# ── Correlation heatmap: selected universe ─────────────────────────────────

corr_matrix = uni_returns.dropna(how="any").corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr_matrix.values, cmap="RdBu", vmin=-1, vmax=1, aspect="auto")
ticks = range(len(corr_matrix))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(corr_matrix.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr_matrix.index, fontsize=7)
ax.set_title("Correlation matrix — selected universe")
plt.colorbar(im, ax=ax, fraction=0.025)
plt.tight_layout()
plt.show()

---
## §5 — PCA research

Rolling PCA decomposes the universe into orthogonal factors.  PC1 is
typically the *market factor* — the direction all assets move together.
We track how much variance PC1 explains over time as a proxy for market
cohesion.

In [ ]:
# ── Run rolling PCA for each configured window ────────────────────────────

# Use the standardised return panel (NaN rows dropped per-window inside rolling_pca).
# Fill remaining NaNs with 0 before passing in — they represent assets not yet
# available; a zero return does not distort PCA loadings significantly.
uni_std_filled = uni_returns_std.fillna(0.0)

pca_results = {}
for win in CFG["pca_windows"]:
    print(f"Running rolling PCA: window={win} bars ...")
    pca_results[win] = rolling_pca(
        uni_std_filled,
        window=win,
        n_components=CFG["pca_n_components"],
        standardize=True,
    )
    ev = pca_results[win]["pc1_variance"]
    if not ev.empty:
        print(f"  PC1 variance explained: mean={ev.mean():.3f}, latest={ev.iloc[-1]:.3f}")

# Use the primary window for downstream analysis.
primary_win = CFG["pca_windows"][0]
pca = pca_results[primary_win]
print(f"\nPrimary PCA window: {primary_win} bars")

In [ ]:
# ── Plot: explained variance over time (primary window) ───────────────────

ev_df = pca["explained_variance"]

if not ev_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # Per-PC explained variance.
    for col in ev_df.columns:
        axes[0].plot(ev_df.index, ev_df[col], lw=0.8, label=col)
    axes[0].set_title(f"Rolling explained variance ratio by PC (window={primary_win} bars)")
    axes[0].set_ylabel("EV ratio")
    axes[0].legend(ncol=ev_df.shape[1], fontsize=8)

    # Cumulative explained variance and PC1 alone.
    axes[1].fill_between(
        pca["cumulative_variance"].index,
        pca["cumulative_variance"].values,
        alpha=0.25, color="steelblue", label="Cum EV (all PCs)"
    )
    axes[1].plot(
        pca["pc1_variance"].index,
        pca["pc1_variance"].values,
        color="crimson", lw=1.2, label="PC1 EV"
    )
    axes[1].axhline(0.45, color="orange", ls="--", lw=1, label="cohesion threshold (0.45)")
    axes[1].set_ylabel("Explained variance")
    axes[1].set_title("Rolling PC1 variance explained vs cumulative")
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Plot: latest PC loadings ───────────────────────────────────────────────

loadings_tbl = pca_loadings_table(pca, n_pcs=3, top_n=len(universe_syms))

if not loadings_tbl.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    x = range(len(loadings_tbl))
    width = 0.25
    for i, pc in enumerate(loadings_tbl.columns[:3]):
        ax.bar(
            [xi + i * width for xi in x],
            loadings_tbl[pc].values,
            width=width, label=pc, alpha=0.8,
        )
    ax.set_xticks([xi + width for xi in x])
    ax.set_xticklabels(loadings_tbl.index, rotation=90, fontsize=7)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Latest PC loadings (window={primary_win} bars)")
    ax.set_ylabel("Loading")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Plot: BTC and ETH loadings on PC1 over time ────────────────────────────

fig, axes = plt.subplots(len(CFG["pca_windows"]), 1,
                         figsize=(14, 4 * len(CFG["pca_windows"])),
                         sharex=False)
if len(CFG["pca_windows"]) == 1:
    axes = [axes]

for ax, win in zip(axes, CFG["pca_windows"]):
    res = pca_results[win]
    btc_load = res["btc_loading"]
    eth_load = res["eth_loading"]
    if btc_load.empty:
        ax.set_title(f"BTC/ETH PC1 loading — window={win} bars (no data)")
        continue
    ax.plot(btc_load.index, btc_load.values, lw=1.2, label="BTC PC1 loading", color="steelblue")
    ax.plot(eth_load.index, eth_load.values, lw=1.2, label="ETH PC1 loading", color="darkorange")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_title(f"BTC / ETH PC1 loading over time (window={win} bars)")
    ax.set_ylabel("PC1 loading")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Compare PC1 variance explained across windows ─────────────────────────

fig, ax = plt.subplots(figsize=(14, 4))
for win in CFG["pca_windows"]:
    pc1_ev = pca_results[win]["pc1_variance"]
    if not pc1_ev.empty:
        ax.plot(pc1_ev.index, pc1_ev.values, lw=1.0, label=f"window={win}")
ax.axhline(0.45, color="red", ls="--", lw=1, label="cohesion threshold")
ax.set_title("Rolling PC1 variance explained — comparison across windows")
ax.set_ylabel("EV ratio")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## §6 — K-means clustering

We cluster the universe assets using their PC loadings as features.  Assets
in the same cluster share similar factor exposures.  We sweep `k` from 2 to 8,
inspect the elbow and silhouette plots, then fix `k = CFG["kmeans_k_final"]`.

In [ ]:
# ── Build clustering feature matrix from latest PCA loadings ──────────────

# Use the full loadings_all table: one row per asset, one column per PC.
n_pcs = CFG["pca_n_components"]
load_all = pca["loadings_all"]

feature_rows = {}
for ci in range(n_pcs):
    ldf = load_all.get(ci)
    if ldf is not None and not ldf.empty:
        feature_rows[f"PC{ci+1}"] = ldf.iloc[-1]  # latest snapshot

if feature_rows:
    feat_df    = pd.DataFrame(feature_rows)  # (n_assets, n_pcs)
    feat_arr   = feat_df.values
    feat_syms  = list(feat_df.index)
    print(f"Clustering feature matrix: {feat_arr.shape[0]} assets × {feat_arr.shape[1]} PCs")
else:
    print("WARNING: No PCA loadings available for clustering. Re-check PCA step.")
    feat_arr  = np.empty((0, 0))
    feat_syms = []

In [ ]:
# ── K-means sweep: elbow + silhouette ─────────────────────────────────────

if feat_arr.size > 0:
    sweep_df = kmeans_sweep(
        feat_arr,
        k_range=CFG["kmeans_k_range"],
        random_seed=CFG["random_seed"],
        n_init=CFG["kmeans_n_init"],
    )
    display(sweep_df)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(sweep_df["k"], sweep_df["inertia"], marker="o", color="steelblue")
    axes[0].axvline(CFG["kmeans_k_final"], color="red", ls="--", lw=1,
                    label=f"k_final={CFG['kmeans_k_final']}")
    axes[0].set_xlabel("k")
    axes[0].set_ylabel("Inertia")
    axes[0].set_title("K-means elbow plot")
    axes[0].legend()

    axes[1].plot(sweep_df["k"], sweep_df["silhouette"], marker="o", color="darkorange")
    axes[1].axvline(CFG["kmeans_k_final"], color="red", ls="--", lw=1,
                    label=f"k_final={CFG['kmeans_k_final']}")
    axes[1].set_xlabel("k")
    axes[1].set_ylabel("Silhouette score")
    axes[1].set_title("Silhouette score by k")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Fit final K-means model ────────────────────────────────────────────────

if feat_arr.size > 0:
    k_final = min(CFG["kmeans_k_final"], len(feat_syms) - 1)
    km_final = KMeans(
        n_clusters=k_final,
        random_state=CFG["random_seed"],
        n_init=CFG["kmeans_n_init"],
    )
    cluster_labels = km_final.fit_predict(feat_arr)

    # Print cluster membership.
    for cid in range(k_final):
        members = [s for s, lbl in zip(feat_syms, cluster_labels) if lbl == cid]
        print(f"Cluster {cid}: {members}")

In [ ]:
# ── Cluster analytics ─────────────────────────────────────────────────────

if feat_arr.size > 0:
    # Align return panel to the assets in feat_syms.
    km_rets = uni_returns[[s for s in feat_syms if s in uni_returns.columns]].dropna(how="all")

    # Re-align cluster_labels to match km_rets columns.
    label_map = {s: lbl for s, lbl in zip(feat_syms, cluster_labels)}
    km_labels = np.array([label_map[s] for s in km_rets.columns])

    ca_df = cluster_analytics(
        km_rets,
        labels=km_labels,
        btc_col=btc_col,
        eth_col=eth_col,
        lag_horizons=CFG["lag_horizons"],
    )
    display(ca_df)

In [ ]:
# ── Plot: PCA scatter coloured by cluster ─────────────────────────────────

if feat_arr.size > 0 and feat_arr.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(
        feat_arr[:, 0], feat_arr[:, 1],
        c=cluster_labels, cmap="tab10",
        s=80, alpha=0.85, edgecolors="white", linewidths=0.5,
    )
    for i, sym in enumerate(feat_syms):
        ax.annotate(sym, (feat_arr[i, 0], feat_arr[i, 1]),
                    fontsize=6, ha="center", va="bottom", alpha=0.75)
    ax.set_xlabel("PC1 loading")
    ax.set_ylabel("PC2 loading")
    ax.set_title(f"PCA loading space — K-means clusters (k={k_final})")
    plt.colorbar(scatter, ax=ax, label="cluster id")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Plot: cluster average returns over time ────────────────────────────────

if feat_arr.size > 0:
    fig, ax = plt.subplots(figsize=(14, 5))
    for cid in range(k_final):
        members = [s for s, lbl in zip(feat_syms, cluster_labels)
                   if lbl == cid and s in km_rets.columns]
        if not members:
            continue
        cluster_ret = km_rets[members].mean(axis=1)
        cumret = (1 + cluster_ret).cumprod()
        ax.plot(cumret.index, cumret.values, lw=1.2, label=f"Cluster {cid}")
    ax.set_title("Cluster cumulative return (equal-weight)")
    ax.set_ylabel("Cumulative return (1 = start)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Cluster correlation heatmap ────────────────────────────────────────────

if feat_arr.size > 0:
    cluster_ret_series = {}
    for cid in range(k_final):
        members = [s for s, lbl in zip(feat_syms, cluster_labels)
                   if lbl == cid and s in km_rets.columns]
        if members:
            cluster_ret_series[f"C{cid}"] = km_rets[members].mean(axis=1)

    cluster_ret_df = pd.DataFrame(cluster_ret_series)
    all_corr = pd.concat(
        [cluster_ret_df, km_rets[[btc_col, eth_col]]], axis=1
    ).dropna().corr()

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(all_corr.values, cmap="RdBu", vmin=-1, vmax=1)
    ticks = range(len(all_corr))
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(all_corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(all_corr.index)
    for i in range(len(all_corr)):
        for j in range(len(all_corr)):
            ax.text(j, i, f"{all_corr.values[i,j]:.2f}",
                    ha="center", va="center", fontsize=8,
                    color="white" if abs(all_corr.values[i,j]) > 0.5 else "black")
    ax.set_title("Cluster return correlation heatmap (incl. BTC/ETH)")
    plt.colorbar(im, ax=ax, fraction=0.025)
    plt.tight_layout()
    plt.show()

---
## §7 — BTC/ETH residual model & forward predictive tests

We model BTC/ETH returns as a linear combination of the first N PC returns
(or cluster returns).  The residual is the idiosyncratic part — the piece
not explained by the broad factor structure.  A large positive residual while
the market is flat may indicate early directional pressure.

**Hypothesis tests:**  
Does `btc_residual_z[t]` predict `BTC_return[t+h]` for h = 1, 5, 15, 30?

In [ ]:
# ── Build factor returns (PC scores) ──────────────────────────────────────

pc_scores = pca["pc_scores"]

if not pc_scores.empty:
    # Restrict to first N_FACTORS PCs.
    N_FACTORS = min(3, pc_scores.shape[1])
    factor_returns = pc_scores.iloc[:, :N_FACTORS]

    # Align to universe returns.
    btc_aligned = uni_returns[btc_col].reindex(factor_returns.index)
    eth_aligned = uni_returns[eth_col].reindex(factor_returns.index)

    print(f"Factor returns: {factor_returns.shape[0]:,} rows × {factor_returns.shape[1]} factors")
else:
    print("WARNING: PC scores unavailable. Falling back to cluster returns as factors.")
    factor_returns = cluster_ret_df.reindex(uni_returns.index).dropna(how="all")
    btc_aligned    = uni_returns[btc_col].reindex(factor_returns.index)
    eth_aligned    = uni_returns[eth_col].reindex(factor_returns.index)

In [ ]:
# ── Compute BTC/ETH residuals ──────────────────────────────────────────────

residuals_df = compute_residuals(
    btc_rets=btc_aligned.dropna(),
    eth_rets=eth_aligned.dropna(),
    factor_returns=factor_returns,
    rolling_window=CFG["residual_window"],
)

if not residuals_df.empty:
    print(f"Residual model: {len(residuals_df):,} rows")
    display(residuals_df.dropna().describe().round(5))

In [ ]:
# ── Plot: BTC/ETH residual z-scores and market_beta_pressure ──────────────

if not residuals_df.empty:
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    axes[0].plot(residuals_df.index, residuals_df["btc_residual_z"], lw=0.8,
                 color="steelblue", label="btc_residual_z")
    axes[0].axhline(2, color="red", ls="--", lw=0.8)
    axes[0].axhline(-2, color="red", ls="--", lw=0.8)
    axes[0].set_ylabel("Z-score")
    axes[0].set_title("BTC residual z-score")
    axes[0].legend()

    axes[1].plot(residuals_df.index, residuals_df["eth_residual_z"], lw=0.8,
                 color="darkorange", label="eth_residual_z")
    axes[1].axhline(2, color="red", ls="--", lw=0.8)
    axes[1].axhline(-2, color="red", ls="--", lw=0.8)
    axes[1].set_ylabel("Z-score")
    axes[1].set_title("ETH residual z-score")
    axes[1].legend()

    axes[2].fill_between(
        residuals_df.index, residuals_df["market_beta_pressure"],
        0, where=residuals_df["market_beta_pressure"] > 0,
        alpha=0.5, color="green", label="positive pressure"
    )
    axes[2].fill_between(
        residuals_df.index, residuals_df["market_beta_pressure"],
        0, where=residuals_df["market_beta_pressure"] <= 0,
        alpha=0.5, color="red", label="negative pressure"
    )
    axes[2].set_ylabel("Pressure (-1 to +1)")
    axes[2].set_title("market_beta_pressure")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Forward predictive tests ───────────────────────────────────────────────
# Test whether btc_residual_z and eth_residual_z predict future returns.

if not residuals_df.empty:
    btc_raw_aligned = uni_returns[btc_col].reindex(residuals_df.index)
    eth_raw_aligned = uni_returns[eth_col].reindex(residuals_df.index)

    test_signals = {
        "btc_residual_z": (residuals_df["btc_residual_z"], btc_raw_aligned),
        "eth_residual_z": (residuals_df["eth_residual_z"], eth_raw_aligned),
        "market_beta_pressure": (residuals_df["market_beta_pressure"], btc_raw_aligned),
    }

    # Also test PC1 return against BTC/ETH.
    if not pc_scores.empty:
        pc1_series = pc_scores["PC1"].reindex(residuals_df.index)
        test_signals["pc1_score"] = (pc1_series, btc_raw_aligned)

    all_test_results = {}
    for sig_name, (sig_series, target_series) in test_signals.items():
        result = forward_predictive_test(
            signal=sig_series.dropna(),
            target_rets=target_series,
            horizons=CFG["lag_horizons"],
            n_quantiles=5,
            bootstrap_n=500,
            random_seed=CFG["random_seed"],
        )
        all_test_results[sig_name] = result

    print("Forward predictive test summary:")
    summary_dfs = []
    for sig_name, res in all_test_results.items():
        df_s = summarise_predictive_tests(res, label=sig_name)
        summary_dfs.append(df_s)
    summary_all = pd.concat(summary_dfs, ignore_index=True)
    display(summary_all)

In [ ]:
# ── Plot: quantile response curves for btc_residual_z ─────────────────────

if not residuals_df.empty and "btc_residual_z" in all_test_results:
    results_z = all_test_results["btc_residual_z"]
    valid_horizons = [h for h, r in results_z.items() if r is not None]

    if valid_horizons:
        n_cols = min(4, len(valid_horizons))
        n_rows = (len(valid_horizons) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows), squeeze=False)

        for idx, h in enumerate(valid_horizons):
            ax  = axes[idx // n_cols][idx % n_cols]
            res = results_z[h]
            qmr = res["quantile_mean_returns"]
            if qmr.empty:
                ax.set_title(f"h={h} (insufficient data)")
                continue
            colors = ["green" if v > 0 else "red" for v in qmr.values]
            ax.bar(range(len(qmr)), qmr.values * 1e4, color=colors, alpha=0.8)
            ax.set_xticks(range(len(qmr)))
            ax.set_xticklabels(qmr.index)
            ax.axhline(0, color="black", lw=0.8)
            ax.set_title(
                f"btc_residual_z → BTC return at h={h}\n"
                f"r={res['pearson_r']:.3f}, hit={res['hit_rate']:.2%}"
            )
            ax.set_xlabel("Quantile bucket")
            ax.set_ylabel("Mean future return (bps)")

        # Hide unused axes.
        for idx in range(len(valid_horizons), n_rows * n_cols):
            axes[idx // n_cols][idx % n_cols].set_visible(False)

        plt.suptitle(
            "Quantile response: btc_residual_z → future BTC return",
            y=1.01, fontsize=12,
        )
        plt.tight_layout()
        plt.show()

---
## §8 — Wave feature proposal

The table below catalogues candidate Wave features that could be wired into
the production `WaveEngine` after this research is reviewed.  Each feature
includes a description, computation formula, update cadence, required inputs,
expected range, and intended use in Wave permissions.

**This is output from research — not production code.**

In [ ]:
feat_tbl = wave_feature_table()

# Display with wrapping so formulas and descriptions are readable.
with pd.option_context("display.max_colwidth", 120, "display.max_rows", 50):
    display(feat_tbl)

In [ ]:
# ── Empirical feature values at the latest timestamp ──────────────────────

if not residuals_df.empty and not pc_scores.empty:
    latest = residuals_df.dropna(subset=["btc_residual_z", "eth_residual_z"]).iloc[-1]
    latest_pc = pc_scores.iloc[-1]
    latest_pc1_ev = pca["pc1_variance"].iloc[-1] if not pca["pc1_variance"].empty else np.nan
    market_cohesion = uni_returns.dropna(how="any").corr().values
    # Mean off-diagonal correlation as a simple cohesion proxy.
    n = len(market_cohesion)
    cohesion = (
        (market_cohesion.sum() - n) / (n * (n - 1))
        if n > 1 else np.nan
    )

    snapshot = {
        "wave.pc1_return":             round(float(latest_pc["PC1"]), 6),
        "wave.pc1_variance_explained": round(float(latest_pc1_ev), 4),
        "wave.market_cohesion":        round(float(cohesion), 4),
        "wave.btc_residual_z":         round(float(latest["btc_residual_z"]), 4),
        "wave.eth_residual_z":         round(float(latest["eth_residual_z"]), 4),
        "wave.market_beta_pressure":   round(float(latest["market_beta_pressure"]), 4),
    }
    print("Latest empirical feature values:")
    for k, v in snapshot.items():
        print(f"  {k:<40} {v}")

---
## §9 — Regime interpretation (research only)

> **Important:** The rule mapping below is exploratory.  It is NOT production
> trading logic and should not be connected to the live trading path without
> a full review and validation cycle.

We translate combinations of the proposed features into candidate Wave regime
labels.  The goal is to build intuition for how the multi-asset features could
augment the single-symbol Wave regime classification.

In [ ]:
# ── Regime rules table ─────────────────────────────────────────────────────

rules_tbl = regime_rules_table()
with pd.option_context("display.max_colwidth", 120):
    display(rules_tbl)

In [ ]:
# ── Exploratory regime labelling over the residual time series ────────────
# Apply simple threshold rules to the computed features and label each bar.
# Intended only for visual inspection — not production logic.

if not residuals_df.empty:
    pc1_ev_ts = pca["pc1_variance"].reindex(residuals_df.index).fillna(method="ffill")
    mbp        = residuals_df["market_beta_pressure"]

    def _label_regime(row):
        ev  = row.get("pc1_ev", np.nan)
        mbp = row.get("mbp", np.nan)
        if pd.isna(ev) or pd.isna(mbp):
            return "UNKNOWN"
        if ev > 0.45 and mbp > 0.3:
            return "BREAKOUT_RISK_ON"
        if ev > 0.45 and mbp < -0.3:
            return "BREAKOUT_RISK_OFF"
        if ev < 0.30:
            return "FRAGMENTED"
        if -0.2 <= mbp <= 0.2:
            return "NEUTRAL"
        return "DIRECTIONAL_PRESSURE"

    regime_frame = pd.DataFrame({
        "pc1_ev": pc1_ev_ts,
        "mbp":    mbp,
    }).dropna()
    regime_frame["regime"] = regime_frame.apply(_label_regime, axis=1)

    regime_counts = regime_frame["regime"].value_counts()
    print("Regime distribution (research labels):")
    print(regime_counts.to_string())
    print(f"\nTotal bars classified: {len(regime_frame):,}")

In [ ]:
# ── Plot: regime overlay on BTC price ─────────────────────────────────────

if not residuals_df.empty and "regime" in regime_frame.columns:
    btc_price = prices.get(btc_col, pd.Series(dtype=float)).reindex(regime_frame.index)

    regime_color = {
        "BREAKOUT_RISK_ON":     "green",
        "BREAKOUT_RISK_OFF":    "red",
        "FRAGMENTED":           "grey",
        "NEUTRAL":              "steelblue",
        "DIRECTIONAL_PRESSURE": "darkorange",
        "UNKNOWN":              "white",
    }

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # BTC price.
    if not btc_price.dropna().empty:
        axes[0].plot(btc_price.index, btc_price.values, lw=0.8, color="black")
        axes[0].set_title("BTC price with research regime overlay")
        axes[0].set_ylabel("BTC price")
        # Shade regime bands.
        prev_idx = regime_frame.index[0]
        prev_regime = regime_frame["regime"].iloc[0]
        for i in range(1, len(regime_frame)):
            cur_regime = regime_frame["regime"].iloc[i]
            if cur_regime != prev_regime or i == len(regime_frame) - 1:
                axes[0].axvspan(
                    prev_idx, regime_frame.index[i],
                    alpha=0.15,
                    color=regime_color.get(prev_regime, "white"),
                )
                prev_idx   = regime_frame.index[i]
                prev_regime = cur_regime

    # Regime colour strip.
    regime_int = regime_frame["regime"].map(
        {r: i for i, r in enumerate(regime_color)}
    ).fillna(0)
    axes[1].scatter(
        regime_frame.index, np.ones(len(regime_frame)),
        c=[list(regime_color.values())[int(v)] for v in regime_int],
        marker="|", s=50, linewidths=1,
    )
    # Legend.
    for label, color in regime_color.items():
        axes[1].scatter([], [], color=color, label=label, marker="s")
    axes[1].legend(ncol=3, fontsize=8, loc="upper left")
    axes[1].set_yticks([])
    axes[1].set_title("Research regime strip (NOT production logic)")

    plt.tight_layout()
    plt.show()

---
## Summary & next steps

### What this notebook established

| Finding | Implication |
|---|---|
| Universe pruning | Ranked tables identify which non-BTC/ETH assets carry the most explanatory information for BTC/ETH returns. |
| PC1 variance explained | A time-varying measure of market cohesion.  High values coincide with directional, correlated markets. |
| BTC/ETH PC1 loadings | Confirm BTC/ETH are typically the dominant market-factor assets. |
| K-means clusters | Stable groupings emerge (e.g. large-cap crypto, macro/FX, high-beta altcoins) with distinct BTC/ETH correlation profiles. |
| Residual z-scores | `btc_residual_z` and `eth_residual_z` quantify idiosyncratic pressure not explained by the broad factor. |
| Forward predictive tests | Quantile and hit-rate tests reveal whether the residual model carries short-horizon predictive value. |

### Proposed Wave features

See §8 for the full catalogue.  Priority candidates for production implementation:

1. `wave.pc1_variance_explained` — immediate cohesion context signal.
2. `wave.market_beta_pressure` — directional composite for permission weighting.
3. `wave.btc_residual_z` / `wave.eth_residual_z` — idiosyncratic pressure.
4. `wave.cluster_leader_return` — early directional signal from leading cluster.

### TODO before production

- [ ] Validate that collected Binance + Oanda data covers the required date range.
- [ ] Review exact column names for Oanda instruments (check `list_ohlcv('oanda', '1m')` output).
- [ ] Choose rolling window sizes based on regime-conditional performance.
- [ ] Run out-of-sample validation on the residual model.
- [ ] Review K-means stability across different time periods.
- [ ] Design production update cadence for each feature (Wave tick = ~5 s).
- [ ] Wire validated features into `WaveFeatureParams` and `WaveEngine` after full review.

> **Scope reminder:** Do not modify the live trading path based on this notebook.
> This is research output only.